# Phase 6 — Google Colab Model Training
## Dual-Stream SAR + Sentinel-2 Building Damage Detection

This notebook trains a supervised 3-class building-level damage detection model using fused Sentinel-1 (SAR) and Sentinel-2 (optical) pre/post imagery.

## Part 1 — Google Colab Environment Setup

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import torchvision

# 1. Detect GPU
cuda_available = torch.cuda.is_available()
print("CUDA Available:", cuda_available)
if cuda_available:
    print("GPU Device Name:", torch.cuda.get_device_name(0))

# 2. Print environment info
print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", torchvision.__version__)

# 3. Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# 4. Create required directories
os.makedirs("models", exist_ok=True)
os.makedirs("results/predictions", exist_ok=True)
print("Required directories created successfully.")

## Part 2 — Data Access Options

In [ ]:
# Configurable data loading options
USE_GOOGLE_DRIVE = False  # Set to True if mounting Google Drive
USE_ZIP_ARCHIVE = True    # Set to True if extracting a prepared ZIP archive

dataset_path = "data/phase6"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = "/content/drive/MyDrive/phase6_training_data"

if USE_ZIP_ARCHIVE and os.path.exists("data/phase6/phase6_training_data.zip"):
    import zipfile
    print("Extracting training data archive...")
    with zipfile.ZipFile("data/phase6/phase6_training_data.zip", "r") as zip_ref:
        zip_ref.extractall("data/phase6_extracted")
    dataset_path = "data/phase6_extracted"

print("Dataset path configured to:", os.path.abspath(dataset_path))

## Part 3 — Dataset Verification and Critical Stop Conditions

In [ ]:
import pandas as pd
from pathlib import Path

index_csv_path = Path(dataset_path) / "xbd_training_index.csv"

# Verification Check 1: Check if the index CSV file exists
if not index_csv_path.exists():
    raise FileNotFoundError("Building-level supervised labels are missing; training cannot be performed correctly from metadata counts alone.")

df_index = pd.read_csv(index_csv_path)

# Verification Check 2: Verify presence of individual building-level label columns
required_cols = ["sample_id", "xbd_uid", "damage_class", "split", "building_mask_path"]
for col in required_cols:
    if col not in df_index.columns:
        raise ValueError(f"Missing required column: {col} in dataset index.")

# Verification Check 3: STOP if actual building-level annotation/numpy files are missing
# Since we generated a simulated fallback index, we check if the actual building numpy files exist
sample_row = df_index.iloc[0]
mask_file = Path(sample_row["building_mask_path"])

# Check if the building mask files physically exist on disk
if not mask_file.exists():
    error_msg = "Building-level supervised labels are missing; training cannot be performed correctly from metadata counts alone."
    print("\n" + "!"*80)
    print("CRITICAL STOP CONDITION TRIGGERED:")
    print(error_msg)
    print("!"*80 + "\n")
    raise AssertionError(error_msg)

# Dataset Statistics Report
print("TOTAL SAMPLES:", len(df_index))
print("TRAIN:", len(df_index[df_index["split"] == "train"]))
print("VALIDATION:", len(df_index[df_index["split"] == "val"]))
print("TEST:", len(df_index[df_index["split"] == "test"]))

print("\nCLASS DISTRIBUTION:")
print(df_index["damage_label"].value_counts())

## Part 4 — Label Mapping Schema

In [ ]:
"""
Label Mapping Definition:
- no-damage (1)       -> 0 (INTACT)
- minor-damage (2)    -> 1 (DAMAGED)
- major-damage (3)    -> 1 (DAMAGED)
- destroyed (4)       -> 2 (DESTROYED)
- un-classified (5)   -> EXCLUDE
"""
class_mapping = {
    0: "INTACT",
    1: "DAMAGED",
    2: "DESTROYED"
}
print("Label mapping documented and loaded:", class_mapping)

## Part 5 to 12 — Model Preprocessing, Normalization, and Architecture definitions

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 1. Load Normalization stats
norm_json_path = Path(dataset_path) / "normalization.json"
if norm_json_path.exists():
    with open(norm_json_path, "r") as f:
        norm_stats = json.load(f)
    print("Loaded Normalization Parameters:", norm_stats)

# 2. Spatial Augmentation
def apply_spatial_augmentation(sar, optical, mask, p=0.5):
    # Rotate and flip channels identical to prevent spatial mismatch
    if random.random() < p:
        # Horizontal flip
        sar = torch.flip(sar, [2])
        optical = torch.flip(optical, [2])
        mask = torch.flip(mask, [2])
    if random.random() < p:
        # Vertical flip
        sar = torch.flip(sar, [1])
        optical = torch.flip(optical, [1])
        mask = torch.flip(mask, [1])
    return sar, optical, mask

# 3. Dual-Stream Siamese U-Net Model Definition
class DualStreamUNetClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.sar_conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.opt_conv = nn.Sequential(
            nn.Conv2d(8, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(64 + 128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x_sar, x_opt, mask):
        feat_sar = self.sar_conv(x_sar)
        feat_opt = self.opt_conv(x_opt)
        fused = torch.cat([feat_sar, feat_opt], dim=1)
        features = self.fusion(fused)
        
        # Masked Average Pooling
        mask_expanded = mask.expand(-1, 64, -1, -1)
        features_masked = features * mask_expanded
        sum_feat = features_masked.sum(dim=(2, 3))
        sum_mask = mask_expanded.sum(dim=(2, 3)).clamp(min=1.0)
        bld_feat = sum_feat / sum_mask
        
        logits = self.classifier(bld_feat)
        return logits

print("Preprocessors and model initialized successfully.")